In [23]:
import polars as pl
from datetime import datetime

pl.Config.set_tbl_rows(100)

polars.config.Config

In [2]:
df = pl.read_parquet("../data/train.parquet")
print(df.shape)
df.head(2)

(44413436, 5)


user_id,item_id,event_type,watch_time,date
u64,i32,cat,i64,datetime[μs]
4829009381441709630,0,"""favorite""",0,2024-12-02 18:03:33
7219091552393371635,1,"""favorite""",0,2024-12-02 14:44:13


In [36]:
df = df \
    .with_columns(
        is_like=pl.when(pl.col("event_type") == "like").then(1).otherwise(0),
        is_favorite=pl.when(pl.col("event_type") == "favorite").then(1).otherwise(0),
    ) \
    .group_by("user_id", "item_id", pl.col("date").dt.date()) \
    .agg(
        dt=pl.col("date").min(),
        views=pl.len(),
        watch_time=pl.col("watch_time").max(),
        is_like=pl.col("is_like").max(),
        is_favorite=pl.col("is_favorite").max(),
    ) \
    .with_columns(
        is_positive=pl \
            .when(pl.col("watch_time") > 60).then(1) \
            .when(pl.col("is_like") == 1).then(1) \
            .when(pl.col("is_favorite") == 1).then(1) \
            .otherwise(0),
    )

print(df.shape)
df.head(2)

(41637896, 9)


user_id,item_id,date,dt,views,watch_time,is_like,is_favorite,is_positive
u64,i32,date,datetime[μs],u32,i64,i32,i32,i32
6617804886732240288,17899,2024-11-29,2024-11-29 18:12:46,3,11,0,0,0
18298054402076325867,524994,2024-12-01,2024-12-01 10:30:18,1,30,0,0,0


In [37]:
train_df = df.filter(pl.col("date") < datetime.fromisoformat("2024-12-03").date())
test_df = df.filter(pl.col("date") == datetime.fromisoformat("2024-12-03").date())
print(train_df.shape)
print(test_df.shape)
train_df.head(2)

(40409221, 9)
(1228675, 9)


user_id,item_id,date,dt,views,watch_time,is_like,is_favorite,is_positive
u64,i32,date,datetime[μs],u32,i64,i32,i32,i32
6617804886732240288,17899,2024-11-29,2024-11-29 18:12:46,3,11,0,0,0
18298054402076325867,524994,2024-12-01,2024-12-01 10:30:18,1,30,0,0,0


In [45]:
item_features_df = train_df \
    .group_by("item_id", "user_id").agg(
        pl.col("views").sum(),
        pl.col("watch_time").max(),
        pl.col("is_like").max(),
        pl.col("is_favorite").max(),
        pl.col("is_positive").max(),
    ) \
    .group_by("item_id").agg(
        views=pl.col("views").sum(),
        avg_views=pl.col("views").mean(),
        avg_watch_time=pl.col("watch_time").mean(),
        p3_watch_time=pl.col("watch_time").quantile(0.3),
        p8_watch_time=pl.col("watch_time").quantile(0.8),
        median_watch_time=pl.col("watch_time").median(),
        likes=pl.col("is_like").sum(),
        favorites=pl.col("is_favorite").sum(),
        positives=pl.col("is_positive").sum(),
        like_rate=pl.col("is_like").mean(),
        favorite_rate=pl.col("is_favorite").mean(),
        positive_rate=pl.col("is_positive").mean(),
    )

print(item_features_df.shape)
item_features_df.head(2)

(1860887, 13)


item_id,views,avg_views,avg_watch_time,p3_watch_time,p8_watch_time,median_watch_time,likes,favorites,positives,like_rate,favorite_rate,positive_rate
i32,u32,f64,f64,f64,f64,f64,i32,i32,i32,f64,f64,f64
254311,14,1.076923,158.307692,32.0,162.0,71.0,0,1,7,0.0,0.076923,0.538462
13916,68,1.114754,72.245902,32.0,100.0,46.0,5,2,26,0.081967,0.032787,0.42623


In [46]:
user_features_df = train_df \
    .group_by("item_id", "user_id").agg(
        pl.col("views").sum(),
        pl.col("watch_time").max(),
        pl.col("is_like").max(),
        pl.col("is_favorite").max(),
        pl.col("is_positive").max(),
    ) \
    .group_by("user_id").agg(
        views=pl.col("views").sum(),
        avg_views=pl.col("views").mean(),
        avg_watch_time=pl.col("watch_time").mean(),
        p3_watch_time=pl.col("watch_time").quantile(0.3),
        p8_watch_time=pl.col("watch_time").quantile(0.8),
        median_watch_time=pl.col("watch_time").median(),
        likes=pl.col("is_like").sum(),
        favorites=pl.col("is_favorite").sum(),
        positives=pl.col("is_positive").sum(),
        like_rate=pl.col("is_like").mean(),
        favorite_rate=pl.col("is_favorite").mean(),
        positive_rate=pl.col("is_positive").mean(),
    )

print(user_features_df.shape)
user_features_df.head(2)

(408025, 13)


user_id,views,avg_views,avg_watch_time,p3_watch_time,p8_watch_time,median_watch_time,likes,favorites,positives,like_rate,favorite_rate,positive_rate
u64,u32,f64,f64,f64,f64,f64,i32,i32,i32,f64,f64,f64
9843000166072019097,73,1.089552,98.074627,50.0,144.0,75.0,0,0,41,0.0,0.0,0.61194
15426501474337513980,121,1.043103,66.75,40.0,93.0,57.5,0,0,53,0.0,0.0,0.456897


In [47]:
train_df.write_parquet("../data/prepared_train_data.parquet")
test_df.write_parquet("../data/prepared_test_data.parquet")
item_features_df.write_parquet("../data/item_features_data.parquet")
user_features_df.write_parquet("../data/user_features_data.parquet")